# 04 — Thử hiệu năng Qwen 8B trên Colab trước lượt Kaggle qua đêm

Chọn **Colab GPU T4 ngay từ đầu → Run all**. Dùng tài khoản Google
đọc được folder VietMedBridge và ghi được data_staging. Không cần
KAGGLE_JSON, Add Input hay notebook 02/03. Giữ phiên Kaggle đang chạy.

Notebook dùng **Google Drive API**, không mount Drive để đọc hàng nghìn
file. Chỉ đọc thư viện JSON đã freeze và tối đa **ba file missing-text**,
lấy tối đa 288 text rồi tách mẫu chọn batch và mẫu đo độc lập theo ID.
Qwen 8B/NF4, tokenizer, max_length, pooling và runtime khóa như Kaggle.

Sau tải model và warm-up, đo hai lượt native batch 16 và hai lượt bộ
tự chọn v2, đảo thứ tự đo, đồng bộ CUDA và kiểm sai số trên mẫu độc lập.
Báo cáo có texts/s, tokens/s, VRAM, OOM khi tuning và cấu hình được chọn.
[Cách đo CUDA theo PyTorch](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html).

Model/file tạm dùng ổ đĩa runtime Colab; báo cáo nhỏ được lưu qua API
vào **VietMedBridge/data_staging/qwen_performance_benchmarks**. Không
tải model/corpus về máy tính cá nhân, không ghi vector vào model_cache,
không sửa task journal, candidate, kế hoạch hoặc coverage.

Nếu Colab cấp GPU khác T4, báo cáo ghi T4 match=False; không suy ra tốc
độ Kaggle T4 từ GPU đó. Đây là phép đo trên mẫu, chưa phải chứng nhận
chất lượng hoặc tốc độ toàn corpus. Chưa tự thay cấu hình Kaggle.

Sau khi có BENCHMARK_SAVED_ON_DRIVE, báo cáo đã lưu để đọc và chốt lượt
tiếp theo. Chạy lại cell 4 chỉ lưu báo cáo, không chạy lại model. Cùng
runtime có thể dùng lại kết quả đo; bật FORCE_REMEASURE để chủ động đo mới.

**Cell 5–6 thử các nhánh bổ sung so trực tiếp với v2:** batch 8/12/24,
CPU thread 1, và efficient SDPA với K/V head expansion khi cần. Không
cài kernel ngoài hoặc đổi weights/mask/pooling. Chỉ chọn nhánh nhanh
hơn v2 ít nhất 5% trên hai lượt đo độc lập, đồng thời vẫn đạt sai số
so với encoder gốc. Nếu không có nhánh nào vượt vòng chọn, bỏ qua
đo lặp trên mẫu độc lập để tiết kiệm GPU và giữ v2.
[Giới hạn GQA của PyTorch 2.8](https://docs.pytorch.org/docs/2.8/generated/torch.nn.functional.scaled_dot_product_attention.html).

Cell 5–6 chỉ thử nghiệm; chưa thay worker Kaggle hoặc tính task mới.
Phép đo cũ được dùng lại nếu còn trong runtime. Báo cáo bổ sung có
T4_EXTRA_SUMMARY và BENCHMARK_SAVED_ON_DRIVE riêng.

In [ ]:
#@title 1. Colab GPU T4 — Drive API, thư viện frozen và runtime Kaggle
import copy,hashlib,importlib,importlib.metadata,json,os,re,shutil,subprocess,sys,time
from pathlib import Path
if not Path('/var/colab/hostname').is_file() or not shutil.which('nvidia-smi'):
    raise RuntimeError('Chọn Colab GPU T4 ngay từ đầu rồi Run all; không chạy giai đoạn CPU trước.')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=True)
WORK=Path('/content/vmb_qwen_colab_benchmark')
WORK.mkdir(parents=True,exist_ok=True)
FORCE_REMEASURE=False #@param {type:"boolean"}
sys.path.insert(0,str(WORK))
from google.colab import auth
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
SOURCES=json.loads('{"new_data_drive_api.py": "\\"\\"\\"API receipt reads and narrowly scoped lookup checkpoints, without Drive FUSE.\\"\\"\\"\\nimport copy\\nimport hashlib\\nimport json\\nfrom pathlib import Path, PurePosixPath\\nimport re\\nimport threading\\nimport time\\n\\n\\nclass LookupPublishUnavailable(RuntimeError):\\n    \\"\\"\\"An optional lookup upload failed; paid vector receipts remain untouched.\\"\\"\\"\\n\\n\\nclass RemoteReceipt:\\n    def __init__(self, source, metadata):\\n        self.source, self.metadata = source, metadata\\n        self.name = metadata[\'name\']\\n\\n    def read_json(self):\\n        return self.source.read_json(self.metadata)\\n\\n\\nclass DriveReceiptSource:\\n    endpoint = \'https://www.googleapis.com/drive/v3/files\'\\n\\n    def __init__(self, session_factory, data_root, canonical_root, *,\\n                 data_folder_id=\'1Ez1S9jk6FO_puxC8r2vPSc0LJbuOkkdj\', report=print):\\n        if not re.fullmatch(r\'[A-Za-z0-9_-]+\', data_folder_id):\\n            raise ValueError(\'Invalid Drive data folder ID.\')\\n        self.session_factory, self.sessions = session_factory, threading.local()\\n        self.requested_root, self.data_root = Path(data_root).absolute(), Path(canonical_root)\\n        self.folder_id, self.report, self.folders = data_folder_id, report, {}\\n        self.payload_directories = {}\\n\\n    def directory_files(self, relative_directory, *, refresh=False):\\n        \\"\\"\\"One paginated metadata listing, rather than one lookup per payload.\\"\\"\\"\\n        relative = PurePosixPath(relative_directory)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_directory or not relative.parts:\\n            raise ValueError(\'Invalid payload directory.\')\\n        if not refresh and relative_directory in self.payload_directories:\\n            return self.payload_directories[relative_directory]\\n        parent = self.folder_id\\n        for part in relative.parts:\\n            if (parent, part) not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return {}\\n                self.folders[parent, part] = value[\'id\']\\n            parent = self.folders[parent, part]\\n        result = {}\\n        for value in self._list(f\\"\'{parent}\' in parents and trashed=false\\"):\\n            if value[\'name\'] in result:\\n                raise ValueError(\'Duplicate payload names in Drive directory.\')\\n            result[value[\'name\']] = value\\n        self.payload_directories[relative_directory] = result\\n        return result\\n\\n    def payload_metadata(self, relative_path):\\n        relative = PurePosixPath(relative_path)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_path or len(relative.parts) < 2:\\n            raise ValueError(\'Invalid immutable payload path.\')\\n        directory = relative.parent.as_posix()\\n        result = self.directory_files(directory).get(relative.name)\\n        if result is None:\\n            result = self.directory_files(directory, refresh=True).get(relative.name)\\n        if result is None or \'size\' not in result:\\n            raise ValueError(\'Expected immutable payload is missing from Drive.\')\\n        return result\\n\\n    def _session(self):\\n        if not hasattr(self.sessions, \'session\'):\\n            self.sessions.session = self.session_factory()\\n        return self.sessions.session\\n\\n    def _get(self, url, params, *, stream=False):\\n        import requests\\n        session = self._session()\\n        for attempt in range(4):\\n            try:\\n                response = session.get(url, params=params, stream=stream, timeout=(10,60))\\n            except requests.RequestException:\\n                if attempt == 3:\\n                    raise\\n            else:\\n                if response.status_code not in (429,500,502,503,504) or attempt == 3:\\n                    response.raise_for_status()\\n                    return response\\n                response.close()\\n            self.report(f\'Drive receipt API: transient request failure; retry {attempt+1}/3\')\\n            time.sleep(2**attempt)\\n\\n    def _list(self, query):\\n        token, seen = None, set()\\n        while True:\\n            params = {\'q\': query, \'pageSize\': 1000, \'supportsAllDrives\': \'true\',\\n                \'includeItemsFromAllDrives\': \'true\',\\n                \'fields\': \'nextPageToken,incompleteSearch,files(id,name,mimeType,size,md5Checksum)\'}\\n            if token:\\n                params[\'pageToken\'] = token\\n            with self._get(self.endpoint, params) as response:\\n                page = response.json()\\n            if page.get(\'incompleteSearch\'):\\n                raise ValueError(\'Drive receipt listing is incomplete; refusing partial inventory.\')\\n            yield from page.get(\'files\', [])\\n            token = page.get(\'nextPageToken\')\\n            if not token:\\n                return\\n            if token in seen:\\n                raise ValueError(\'Drive receipt pagination repeated; inventory is incomplete.\')\\n            seen.add(token)\\n\\n    def child(self, parent, name, *, folder=False):\\n        safe = name.replace(\'\\\\\\\\\', \'\\\\\\\\\\\\\\\\\').replace(\\"\'\\", \\"\\\\\\\\\'\\")\\n        query = f\\"\'{parent}\' in parents and trashed=false and name=\'{safe}\'\\"\\n        if folder:\\n            query += \\" and mimeType=\'application/vnd.google-apps.folder\'\\"\\n        values = list(self._list(query))\\n        if len(values) > 1:\\n            raise ValueError(\'Duplicate Drive path names; refusing ambiguous receipt source.\')\\n        return values[0] if values else None\\n\\n    def relative(self, name):\\n        relative = PurePosixPath(name)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in name or not relative.parts:\\n            raise ValueError(\'Invalid relative Drive receipt path.\')\\n        parent = self.folder_id\\n        for part in relative.parts[:-1]:\\n            key = parent, part\\n            if key not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return None\\n                self.folders[key] = value[\'id\']\\n            parent = self.folders[key]\\n        return self.child(parent, relative.name)\\n\\n    def download(self, metadata, target, *, expected_sha256=None, max_bytes=None):\\n        size = int(metadata[\'size\'])\\n        if max_bytes is not None and size > max_bytes:\\n            raise ValueError(\'Receipt exceeds the bounded JSON size.\')\\n        sha, md5, count = hashlib.sha256(), hashlib.md5(), 0\\n        with self._get(self.endpoint+\'/\'+metadata[\'id\'], {\'alt\':\'media\',\'supportsAllDrives\':\'true\'}, stream=True) as response:\\n            with Path(target).open(\'wb\') as output:\\n                for chunk in response.iter_content(chunk_size=1024*1024):\\n                    count += len(chunk)\\n                    if count > size or (max_bytes is not None and count > max_bytes):\\n                        raise ValueError(\'Drive receipt payload exceeds its declared size.\')\\n                    sha.update(chunk)\\n                    md5.update(chunk)\\n                    output.write(chunk)\\n        if (count != size or (metadata.get(\'md5Checksum\') and md5.hexdigest() != metadata[\'md5Checksum\'])\\n            or (expected_sha256 and sha.hexdigest() != expected_sha256)):\\n            raise ValueError(\'Drive receipt size/checksum mismatch.\')\\n\\n    def read_json(self, metadata):\\n        import tempfile\\n        with tempfile.TemporaryDirectory(prefix=\'vmb-api-json-\') as directory:\\n            path = Path(directory) / \'receipt.json\'\\n            self.download(metadata, path, max_bytes=16*1024*1024)\\n            return json.loads(path.read_text(encoding=\'utf-8\'))\\n\\n    def relative_json(self, path):\\n        metadata = self.relative(path)\\n        return None if metadata is None else self.read_json(metadata)\\n\\n    def verify_candidate(self, expected):\\n        active = self.relative_json(\'active_data_candidate.json\')\\n        if active is None or active.get(\'candidate_manifest_sha256\') != expected:\\n            raise ValueError(\'Drive API data folder differs from the expected frozen candidate.\')\\n        candidate = self.relative_json(\'processed/\'+active[\'build_run\']+\'/\'+active[\'candidate_name\'])\\n        from vietmedbridge.artifacts import digest_json\\n        if (candidate is None or candidate.get(\'candidate_manifest_sha256\') != expected\\n            or digest_json({k:v for k,v in candidate.items() if k != \'candidate_manifest_sha256\'}) != expected):\\n            raise ValueError(\'Drive API candidate digest changed.\')\\n        self.active, self.candidate = active, candidate\\n\\n    def receipts(self, identity):\\n        from vietmedbridge.artifacts import digest_json\\n        # Resolve the folder itself without relying on any receipt being present.\\n        cache = self.child(self.folder_id, \'model_cache\', folder=True)\\n        folder = self.child(cache[\'id\'], digest_json(identity)[:24], folder=True) if cache else None\\n        if folder is None:\\n            return []\\n        result, names, payloads = [], set(), {}\\n        # Drive name-contains is prefix matching, not a suffix predicate. List\\n        # the complete producer folder then filter receipt names locally.\\n        query = f\\"\'{folder[\'id\']}\' in parents and trashed=false\\"\\n        for value in self._list(query):\\n            if value[\'name\'] in payloads:\\n                raise ValueError(\'Duplicate names in content cache directory.\')\\n            payloads[value[\'name\']] = value\\n            if not value[\'name\'].endswith(\'.done.json\'):\\n                continue\\n            if (not re.fullmatch(r\'block-[0-9a-f]{64}\\\\.done\\\\.json\', value[\'name\'])\\n                or value[\'name\'] in names or \'size\' not in value):\\n                raise ValueError(\'Invalid or duplicate API receipt filename.\')\\n            names.add(value[\'name\'])\\n            result.append(RemoteReceipt(self, value))\\n        self.payload_directories[\'model_cache/\'+digest_json(identity)[:24]] = payloads\\n        self.report(f\'Drive receipt API: complete listing, {len(result):,} receipts; no mount reads\')\\n        return sorted(result, key=lambda value: value.name)\\n\\n    def publish_cache_checkpoint(self, relative_marker, payload, marker):\\n        \\"\\"\\"Only mapping bundles and exact ANN training samples may be published here.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        binding = marker.get(\'binding\', {})\\n        source = binding.get(\'source_manifest_sha256\', \'\')\\n        kind = binding.get(\'kind\')\\n        if kind == \'mapping-bundle-v1\':\\n            directory, prefix, suffix = \'retrieval/mapping_bundles/\'+source, \'maps-\', \'.tar.gz\'\\n        elif kind == \'index-sample-v1\' and re.fullmatch(r\'[0-9a-f]{64}\', binding.get(\'positions_sha256\', \'\')):\\n            directory = \'retrieval/index_samples/\'+source+\'/\'+binding[\'positions_sha256\']\\n            prefix, suffix = \'sample-\', \'.npy\'\\n        else:\\n            raise ValueError(\'Unsupported cache checkpoint kind.\')\\n        if (binding.get(\'data_folder_id\') != self.folder_id or binding.get(\'schema_version\') != 1\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', source)\\n            or relative_marker != directory+\'/current.json\'\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', marker.get(\'sha256\', \'\'))\\n            or marker.get(\'path\') != prefix+marker[\'sha256\']+suffix\\n            or marker.get(\'manifest_sha256\') != digest_json({k:v for k,v in marker.items() if k != \'manifest_sha256\'})):\\n            raise ValueError(\'Cache checkpoint escapes its declared source/folder binding.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk); md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Cache checkpoint payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def publish_lookup(self, relative_marker, payload, marker):\\n        \\"\\"\\"Upload a checksum-addressed lookup, then its pointer. Never write vector/source files.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        body = {k:v for k,v in marker.items() if k != \'manifest_sha256\'}\\n        binding = marker.get(\'binding\', {})\\n        expected_directory = \'retrieval/content_lookups/\' + digest_json(binding)[:16]\\n        if (relative_marker != expected_directory + \'/current.json\'\\n            or binding.get(\'schema_version\') != 2 or binding.get(\'data_folder_id\') != self.folder_id\\n            or marker.get(\'manifest_sha256\') != digest_json(body)\\n            or marker.get(\'path\') != \'lookup-\' + marker.get(\'sha256\', \'\') + \'.sqlite.gz\'\\n            or not re.fullmatch(r\'lookup-[0-9a-f]{64}\\\\.sqlite\\\\.gz\', marker.get(\'path\', \'\'))):\\n            raise ValueError(\'Lookup upload is outside its bound checkpoint directory.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk)\\n                md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Lookup payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(expected_directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            # Do not include provider messages/URLs: resumable upload URLs are private.\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def _publish_lookup_files(self, directory, payload, marker, md5):\\n        from vietmedbridge.artifacts import digest_json\\n        from googleapiclient.discovery import build\\n        from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload\\n        import io\\n        service = build(\'drive\', \'v3\', credentials=copy.copy(self._session().credentials), cache_discovery=False)\\n        files = service.files()\\n        fields = \'id,name,mimeType,size,md5Checksum\'\\n\\n        def new_id():\\n            return files.generateIds(count=1, space=\'drive\', type=\'files\').execute(num_retries=3)[\'ids\'][0]\\n\\n        def ensure_folder(parent, name):\\n            value = self.child(parent, name, folder=True)\\n            if value is None:\\n                identifier = new_id()\\n                try:\\n                    value = files.create(body={\'id\':identifier, \'name\':name, \'parents\':[parent],\\n                        \'mimeType\':\'application/vnd.google-apps.folder\'}, fields=fields,\\n                        supportsAllDrives=True).execute(num_retries=3)\\n                except Exception:\\n                    # A lost response may follow a successful create. Reconcile the\\n                    # preallocated ID; never retry a folder with a different ID.\\n                    value = files.get(fileId=identifier, fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n                    if value.get(\'name\') != name or value.get(\'mimeType\') != \'application/vnd.google-apps.folder\':\\n                        raise ValueError(\'Created lookup folder identity changed.\')\\n            self.folders[parent, name] = value[\'id\']\\n            return value[\'id\']\\n\\n        parent = self.folder_id\\n        for name in PurePosixPath(directory).parts:\\n            parent = ensure_folder(parent, name)\\n        current = self.child(parent, \'current.json\')\\n        if current is not None:\\n            existing = self.read_json(current)\\n            if (existing.get(\'manifest_sha256\') != digest_json(\\n                    {k:v for k,v in existing.items() if k != \'manifest_sha256\'})\\n                or existing.get(\'binding\') != marker[\'binding\']):\\n                raise ValueError(\'Existing lookup checkpoint binding/digest changed.\')\\n            count_key = \'receipts\' if \'receipts\' in marker else \'parts\'\\n            if existing[count_key] > marker[count_key]:\\n                return existing  # An older local prefix must not replace a newer saved lookup.\\n\\n        def upload(media, name, existing=None):\\n            if existing is None:\\n                request = files.create(body={\'id\':new_id(), \'name\':name, \'parents\':[parent]},\\n                    media_body=media, fields=fields, supportsAllDrives=True)\\n            else:\\n                request = files.update(fileId=existing[\'id\'], media_body=media,\\n                    fields=fields, supportsAllDrives=True)\\n            result = None\\n            while result is None:\\n                _, result = request.next_chunk(num_retries=3)\\n            # Query provider metadata independently before publishing a pointer.\\n            return files.get(fileId=result[\'id\'], fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n\\n        saved = self.child(parent, marker[\'path\'])\\n        if saved is None:\\n            mime = \'application/gzip\' if marker[\'path\'].endswith(\'.gz\') else \'application/octet-stream\'\\n            saved = upload(MediaFileUpload(str(payload), mimetype=mime,\\n                chunksize=8*1024*1024, resumable=True), marker[\'path\'])\\n        if int(saved.get(\'size\', -1)) != payload.stat().st_size or saved.get(\'md5Checksum\') != md5:\\n            raise ValueError(\'Uploaded lookup payload size/checksum mismatch; pointer not advanced.\')\\n        data = json.dumps(marker, ensure_ascii=False, sort_keys=True).encode(\'utf-8\')\\n        saved_marker = upload(MediaIoBaseUpload(io.BytesIO(data), mimetype=\'application/json\',\\n            chunksize=256*1024, resumable=True), \'current.json\', current)\\n        if int(saved_marker.get(\'size\', -1)) != len(data) or saved_marker.get(\'md5Checksum\') != hashlib.md5(data).hexdigest():\\n            raise ValueError(\'Uploaded lookup pointer size/checksum mismatch.\')\\n        return marker\\n\\n\\ndef authenticate_drive_receipts(data_root, expected_candidate_sha256):\\n    \\"\\"\\"Authenticate on the main Colab thread; never print/save credential tokens.\\"\\"\\"\\n    from google.colab import auth\\n    import google.auth\\n    from google.auth.transport.requests import AuthorizedSession\\n    root = Path(data_root).resolve()\\n    print(\'Drive receipts: authenticate API; receipts read-only, lookup checkpoints saved in retrieval/content_lookups\', flush=True)\\n    auth.authenticate_user()\\n    credentials, _ = google.auth.default()\\n    client = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data_root, root)\\n    client.verify_candidate(expected_candidate_sha256)\\n    print(\'Drive receipts: API source verified against the same frozen candidate\', flush=True)\\n    return client\\n", "qwen_token_batching.py": "\\"\\"\\"Measured execution-only batching around the immutable corpus encoder.\\n\\nNo weights, tokenizer options, text, pooling, vector dimensions or cache identity\\nare changed. The native forward is used, with a bounded on-device canary before\\nselecting any faster batching order. There is no local model inference here.\\n\\"\\"\\"\\nfrom contextlib import contextmanager\\nimport gc\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom vietmedbridge.artifacts import atomic_json\\nfrom vietmedbridge.embeddings import validate_vectors\\n\\n\\nPOLICY = \'qwen-token-batches-v2\'\\n\\n\\ndef token_groups(lengths, cap, budget):\\n    order = sorted(range(len(lengths)), key=lambda i: (lengths[i], i))\\n    group = []\\n    for i in order:\\n        if group and (len(group) >= cap or (len(group)+1)*lengths[i] > budget):\\n            yield group\\n            group = []\\n        group.append(i)\\n    if group:\\n        yield group\\n\\n\\ndef vector_agreement(reference, candidate):\\n    if reference.shape != candidate.shape or not np.isfinite(candidate).all():\\n        return False, {\'reason\':\'shape_or_finite\'}\\n    cosine = np.sum(reference*candidate, axis=1) / np.maximum(\\n        np.linalg.norm(reference,axis=1)*np.linalg.norm(candidate,axis=1), 1e-12)\\n    minimum = float(cosine.min())\\n    maximum = float(np.max(np.abs(reference-candidate)))\\n    return minimum >= 0.99999 and maximum <= 0.001, {\'min_cosine\':minimum,\'max_abs_error\':maximum}\\n\\n\\nclass TokenBatchEncoder:\\n    def __init__(self, native, report_path, *, deadline=None, clock=time.perf_counter, report=print,\\n                 attention_tuning=False):\\n        self.native = native\\n        self.encoder = getattr(native,\'encoder\',native)\\n        self.identity, self.dimension = native.identity, native.dimension\\n        self.path = Path(report_path)\\n        self.deadline, self.clock, self.report = deadline, clock, report\\n        self.selected, self.safe_batch_size = None, 16\\n        self.selected_kernel,self.selected_budget = \'native\',24_576\\n        self.attention_tuning = attention_tuning\\n        self.caps, self.ooms = {}, 0\\n        self.attempted = False\\n\\n    def _attention_api(self):\\n        backend=getattr(getattr(self.encoder.torch,\'backends\',None),\'cuda\',None)\\n        setter=getattr(backend,\'allow_fp16_bf16_reduction_math_sdp\',None)\\n        getter=getattr(backend,\'fp16_bf16_reduction_math_sdp_allowed\',None)\\n        return (setter,getter) if callable(setter) and callable(getter) else (None,None)\\n\\n    @contextmanager\\n    def _attention(self,kernel):\\n        if kernel==\'native\':\\n            yield\\n            return\\n        setter,getter=self._attention_api()\\n        if kernel!=\'fp16_math\' or setter is None:\\n            raise ValueError(\'Unsupported attention execution policy.\')\\n        previous=getter()\\n        try:\\n            setter(True)\\n            yield\\n        finally:\\n            setter(previous)\\n\\n    def lengths(self, texts):\\n        # Exactly the native tokenizer\'s options, without batch padding.\\n        ids = self.encoder.tokenizer(list(texts),padding=False,truncation=False)[\'input_ids\']\\n        lengths = [len(row) for row in ids]\\n        if any(n > self.encoder.spec[\'max_length\'] for n in lengths):\\n            raise ValueError(\'Qwen input exceeds the frozen max_length; no text was truncated.\')\\n        return lengths\\n\\n    def _forward(self, texts, indices, lengths, kernel):\\n        bucket = max(lengths[i] for i in indices).bit_length()\\n        cap = self.caps.get((kernel,bucket),len(indices))\\n        if len(indices) > cap:\\n            return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\\n                                   for i in range(0,len(indices),cap)],axis=0)\\n        try:\\n            self.encoder.input_role = \'corpus\'\\n            return np.asarray(self.encoder._batch([texts[i] for i in indices]),dtype=np.float32)\\n        except self.encoder.torch.OutOfMemoryError:\\n            if len(indices) == 1:\\n                raise\\n            self.ooms += 1\\n        # Drop the exception/traceback before freeing CUDA allocations.\\n        cap = max(1,len(indices)//2)\\n        self.caps[kernel,bucket] = cap\\n        gc.collect()\\n        self.encoder.torch.cuda.empty_cache()\\n        self.report(f\'QWEN_BATCH_OOM: token bucket {bucket}; retry same texts with batch {cap}\',flush=True)\\n        return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\\n                               for i in range(0,len(indices),cap)],axis=0)\\n\\n    def _run(self, texts, lengths, cap, kernel=\'native\', budget=24_576):\\n        result = np.empty((len(texts),self.dimension),np.float32)\\n        with self._attention(kernel):\\n            for indices in token_groups(lengths,cap,budget):\\n                values = self._forward(texts,indices,lengths,kernel)\\n                validate_vectors(values,len(indices),self.dimension)\\n                result[indices] = values\\n        return result\\n\\n    def _calibrate(self, texts, lengths):\\n        self.attempted = True\\n        started = self.clock()\\n        order = sorted(range(len(texts)),key=lambda i:lengths[i])\\n        indices = sorted({order[int(i)] for i in np.linspace(0,len(order)-1,min(64,len(order)))})\\n        sample = [texts[i] for i in indices]\\n        sample_lengths = [lengths[i] for i in indices]\\n        self.report(f\'QWEN_TUNING: {len(sample)} existing task texts; native batch 16 versus token batches 16/32/64\',flush=True)\\n        # Warm up only existing task texts. These vectors are never published.\\n        self.native.encode(sample[:4],batch_size=16)\\n        tick = self.clock()\\n        reference = np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\\n        baseline = max(self.clock()-tick,1e-9)\\n        validate_vectors(reference,len(sample),self.dimension)\\n        best, trials = baseline, []\\n        candidates=[(16,\'native\',24_576),(32,\'native\',24_576),(64,\'native\',24_576)]\\n        if self.attention_tuning:\\n            candidates.append((64,\'native\',49_152))\\n            setter,getter=self._attention_api()\\n            if setter is not None and not getter():\\n                candidates.insert(1,(32,\'fp16_math\',24_576))\\n                candidates.insert(2,(64,\'fp16_math\',49_152))\\n        for cap,kernel,budget in candidates:\\n            if self.clock()-started > (120 if self.attention_tuning else 90) or (self.deadline is not None and time.monotonic()+180 >= self.deadline):\\n                break\\n            tick = self.clock()\\n            # Include token-length preparation in the candidate measurement.\\n            self.lengths(sample)\\n            try:\\n                values = self._run(sample,sample_lengths,cap,kernel,budget)\\n            except ValueError:\\n                trials.append({\'batch_cap\':cap,\'kernel\':kernel,\'compatible\':False,\'reason\':\'vector_validation_failed\'})\\n                continue\\n            seconds = max(self.clock()-tick,1e-9)\\n            compatible, agreement = vector_agreement(reference,values)\\n            trial = {\'batch_cap\':cap,\'kernel\':kernel,\'max_padded_tokens\':budget,\\n                     \'seconds\':seconds,\'compatible\':compatible,**agreement}\\n            trials.append(trial)\\n            self.report(f\'QWEN_TUNING_TRIAL: {kernel} cap {cap} | {seconds:.2f}s | compatible {compatible} | native {baseline:.2f}s\',flush=True)\\n            if compatible and seconds < best and seconds <= baseline/1.05:\\n                best, self.selected = seconds, cap\\n                self.selected_kernel,self.selected_budget=kernel,budget\\n        confirmation=None\\n        if self.selected is not None and self.attention_tuning:\\n            # A second comparison prevents choosing a single noisy timing win.\\n            tick=self.clock()\\n            ref2=np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\\n            baseline2=max(self.clock()-tick,1e-9)\\n            tick=self.clock()\\n            self.lengths(sample)\\n            try:\\n                values=self._run(sample,sample_lengths,self.selected,self.selected_kernel,self.selected_budget)\\n                seconds2=max(self.clock()-tick,1e-9)\\n                compatible,agreement=vector_agreement(ref2,values)\\n            except ValueError:\\n                compatible,agreement,seconds2=False,{\'reason\':\'vector_validation_failed\'},baseline2\\n            confirmation={\'compatible\':compatible,\'native_seconds\':baseline2,\'selected_seconds\':seconds2,**agreement}\\n            if not compatible or seconds2>baseline2/1.05:\\n                self.selected,self.selected_kernel,self.selected_budget=None,\'native\',24_576\\n                best=baseline\\n            else:\\n                baseline,best=baseline2,seconds2\\n        self.safe_batch_size = self.selected or 16\\n        record = {\'policy\':POLICY,\'encoder\':self.identity,\'mode\':\'token_batches\' if self.selected else \'native\',\\n            \'sample_texts\':len(sample),\'baseline_seconds\':baseline,\'selected_seconds\':best,\\n            \'batch_cap\':self.safe_batch_size,\'kernel\':self.selected_kernel,\\n            \'max_padded_tokens\':self.selected_budget,\'trials\':trials,\'confirmation\':confirmation,\\n            \'sample_tokens\':sum(sample_lengths),\'sample_token_range\':[min(sample_lengths),max(sample_lengths)],\\n            \'measured_sample_speedup\':baseline/best,\'oom_backoffs\':self.ooms,\\n            \'baseline_texts_per_second\':len(sample)/baseline,\'selected_texts_per_second\':len(sample)/best,\\n            \'scope\':\'Startup canary, not a full-corpus quality or throughput certification.\'}\\n        atomic_json(self.path,record)\\n        self.report(f\'QWEN_TUNING_SELECTED: {record[\\"mode\\"]} {self.selected_kernel} | cap {self.safe_batch_size} | measured sample speedup {baseline/best:.2f}x | {len(sample)/best:.2f} texts/s | report {self.path}\',flush=True)\\n\\n    def encode(self, texts, *, batch_size=16):\\n        if not texts:\\n            return np.empty((0,self.dimension),np.float32)\\n        if not self.attempted and len(texts) >= 32:\\n            self._calibrate(texts,self.lengths(texts))\\n        if self.selected is None:\\n            return self.native.encode(texts,batch_size=16)\\n        lengths=self.lengths(texts)\\n        try:\\n            return self._run(texts,lengths,self.selected,self.selected_kernel,self.selected_budget)\\n        except ValueError:\\n            # Only the current uncommitted task is retried if later inputs expose\\n            # a numerical issue; previous vector blocks remain unchanged.\\n            self.selected,self.selected_kernel,self.safe_batch_size=None,\'native\',16\\n            from vietmedbridge.artifacts import read_json\\n            if self.path.is_file():\\n                saved=read_json(self.path)\\n                saved.update(active_mode=\'native\',runtime_fallback=\'vector_validation_failed\')\\n                atomic_json(self.path,saved)\\n            self.report(\'QWEN_TUNING_FALLBACK: native batch 16 for current uncommitted texts; previous checkpoints retained.\',flush=True)\\n            return self.native.encode(texts,batch_size=16)\\n\\n\\nclass TokenBatchOwner:\\n    def __init__(self, owner, report_path, *, deadline=None, attention_tuning=False):\\n        self.owner, self.path, self.deadline = owner, report_path, deadline\\n        self.attention_tuning=attention_tuning\\n\\n    def for_role(self, role):\\n        native = self.owner.for_role(role)\\n        if role != \'corpus\' or native.identity.get(\'model_id\') != \'Qwen/Qwen3-Embedding-8B\':\\n            return native\\n        return TokenBatchEncoder(native,self.path,deadline=self.deadline,attention_tuning=self.attention_tuning)\\n\\n    def close(self):\\n        self.owner.close()\\n\\n\\ndef benchmark_texts(directory,library):\\n    \\"\\"\\"Sample three bounded Qwen task slices; never walk corpus/vector files.\\"\\"\\"\\n    import pyarrow.parquet as pq\\n    from vietmedbridge.artifacts import digest_json,verify_file\\n    from vietmedbridge.content_embeddings import key\\n    tasks=[t for t in library[\'tasks\'] if t[\'family\']==\'qwen\']\\n    if not tasks:\\n        raise ValueError(\'Library has no Qwen task to benchmark.\')\\n    selected=[tasks[i] for i in sorted({0,len(tasks)//2,len(tasks)-1})]\\n    values=[]\\n    for task in selected:\\n        if Path(task[\'asset\']).name!=task[\'asset\']:\\n            raise ValueError(\'Unsafe benchmark text asset path.\')\\n        path=Path(directory)/task[\'asset\']\\n        verify_file(path,task[\'sha256\'])\\n        table=pq.read_table(path,columns=[\'id\',\'text\'])\\n        if len(table)!=library[\'assets\'][task[\'asset\']][\'rows\']:\\n            raise ValueError(\'Benchmark asset row count changed.\')\\n        rows=table.slice(task[\'offset\'],task[\'rows\']).to_pylist()\\n        if (len(rows)!=task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows))!=task[\'keys_digest\']\\n                or any(r[\'id\']!=key(r[\'text\']) for r in rows)):\\n            raise ValueError(\'Benchmark task text/key binding changed.\')\\n        indices=sorted({int(i) for i in np.linspace(0,len(rows)-1,min(24,len(rows)))})\\n        values.extend(rows[i][\'text\'] for i in indices)\\n    return values\\n\\n\\ndef benchmark(directory,checkout,output,worker_id):\\n    import torch\\n    from full_queue_embeddings import validate_library\\n    from vietmedbridge.artifacts import read_json\\n    from vietmedbridge.portable_embeddings import embedding_runtime\\n    from vietmedbridge.qwen_models import TorchQwenEncoder\\n    from vietmedbridge.qwen_execution import configure_embedding_execution\\n    job=validate_library(directory,checkout=checkout)\\n    if not torch.cuda.is_available() or embedding_runtime()!=job[\'runtime\']:\\n        raise RuntimeError(\'Benchmark must use the original producer CUDA runtime.\')\\n    texts=benchmark_texts(directory,job)\\n    owner=TorchQwenEncoder(job[\'config\'][\'second_dense\'])\\n    try:\\n        native=owner.for_role(\'corpus\')\\n        expected=job[\'models\'][\'qwen\']\\n        if native.identity!=expected[\'encoder\'] or native.dimension!=expected[\'dimension\']:\\n            raise ValueError(\'Benchmark encoder differs from the existing vector producer.\')\\n        configure_embedding_execution(native)\\n        output=Path(output)\\n        path=output/f\'worker-{worker_id}.json\'\\n        tuned=TokenBatchEncoder(native,path,attention_tuning=True,deadline=time.monotonic()+900)\\n        result=tuned.encode(texts,batch_size=64)\\n        validate_vectors(result,len(texts),native.dimension)\\n        report=read_json(path) if path.is_file() else {\'mode\':\'native\',\'reason\':\'not_enough_sample_texts\'}\\n        operators=[]\\n        try:\\n            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as profile:\\n                native.encode([texts[0],texts[-1]],batch_size=2)\\n            operators=sorted({item.key for item in profile.key_averages() if \'scaled_dot_product\' in item.key})\\n        except RuntimeError:\\n            operators=[\'profiler_unavailable\']\\n        report.update(state=\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',job=job[\'manifest_sha256\'],\\n            gpu=torch.cuda.get_device_name(0),worker_id=worker_id,validated_texts=len(texts),\\n            tasks_committed=0,native_sdpa_operators=operators)\\n        atomic_json(path,report)\\n        import json\\n        print(\'QWEN_BENCHMARK_RESULT:\',json.dumps({k:v for k,v in report.items() if k!=\'encoder\'},ensure_ascii=False),flush=True)\\n        return report\\n    finally:\\n        owner.close()\\n\\n\\nif __name__==\'__main__\':\\n    import argparse\\n    parser=argparse.ArgumentParser()\\n    parser.add_argument(\'--job\',required=True)\\n    parser.add_argument(\'--checkout\',required=True)\\n    parser.add_argument(\'--output\',required=True)\\n    parser.add_argument(\'--worker-id\',type=int,required=True)\\n    args=parser.parse_args()\\n    benchmark(args.job,args.checkout,args.output,args.worker_id)\\n", "colab_qwen_benchmark.py": "\\"\\"\\"Bounded Colab Qwen benchmark; never installs vectors or changes task journals.\\"\\"\\"\\nfrom pathlib import Path, PurePosixPath\\nimport argparse\\nimport hashlib\\nimport io\\nimport json\\nimport re\\nimport statistics\\nimport time\\n\\nfrom vietmedbridge.artifacts import atomic_json, digest_json, read_json, sha256_file, utc_now, verify_file\\nfrom vietmedbridge.content_embeddings import checked, key, seal\\n\\nLIBRARY = \'d6ab9a432ac64e5aadb39a826b7dc2176094cc66d4f4e3d9e17c121e071c0711\'\\nSTAGING = \'1UMiIBJBkeFF4cTVif0QPRmjxTUDI_sYg\'\\nSAMPLE_POLICY = \'first-middle-last-96-alternate-v1\'\\n\\n\\ndef benchmark_runtime():\\n    import importlib.metadata\\n    return {name:importlib.metadata.version(name) for name in (\'torch\',\'transformers\',\'bitsandbytes\')}\\n\\n\\ndef validate_job(job):\\n    checked(job)\\n    if (job.get(\'manifest_sha256\') != LIBRARY or job.get(\'schema\') != \'full-data-text-library-v1\'\\n            or job.get(\'repo_url\') != \'https://github.com/Platypus27-coder/VietMedBridge.git\'\\n            or not re.fullmatch(\'[0-9a-f]{40}\', job.get(\'code_commit\', \'\'))\\n            or not job.get(\'input_export_complete\') or job.get(\'coverage_scope\', {}).get(\'official_ids\') != 802028):\\n        raise ValueError(\'Benchmark requires the original frozen 802,028-ID input library.\')\\n    return job\\n\\n\\ndef prepare_sample(source, job, work, *, expected_library=LIBRARY):\\n    \\"\\"\\"Read at most three saved missing-text assets, never the original corpus.\\"\\"\\"\\n    import pyarrow.parquet as pq\\n    checked(job)\\n    if job.get(\'manifest_sha256\') != expected_library:\\n        raise ValueError(\'Benchmark sample belongs to another library.\')\\n    tasks = [t for t in job[\'tasks\'] if t[\'family\'] == \'qwen\' and t[\'rows\'] >= 64]\\n    if not tasks:\\n        raise ValueError(\'No sufficiently sized Qwen task available for a held-out benchmark.\')\\n    selected = [tasks[i] for i in sorted({0, len(tasks)//2, len(tasks)-1})]\\n    work = Path(work)\\n    work.mkdir(parents=True, exist_ok=True)\\n    provenance = [{\'task\':t[\'id\'],\'asset\':t[\'asset\'],\'source_sha256\':t[\'sha256\']} for t in selected]\\n    sample_path = work/\'sample.json\'\\n    if sample_path.is_file():\\n        saved = checked(read_json(sample_path))\\n        if (saved.get(\'job\') != job[\'manifest_sha256\'] or saved.get(\'policy\') != SAMPLE_POLICY\\n                or saved.get(\'source_tasks\') != provenance or saved.get(\'tasks_committed\') != 0):\\n            raise ValueError(\'Saved benchmark sample belongs to another source or policy.\')\\n        print(\'COLAB_BENCHMARK_SAMPLE_REUSED: one small sample JSON; no source asset replay\',flush=True)\\n        return sample_path, saved\\n    tables, values, provenance = {}, {}, []\\n    for task in selected:\\n        name, asset = task[\'asset\'], job[\'assets\'][task[\'asset\']]\\n        relative = PurePosixPath(asset[\'source_path\'])\\n        if (not re.fullmatch(\'missing-qwen-[0-9]{6}[.]parquet\', name) or relative.is_absolute()\\n                or \'..\' in relative.parts or \'\\\\\\\\\' in asset[\'source_path\']\\n                or task[\'sha256\'] != asset[\'sha256\']):\\n            raise ValueError(\'Unsafe or changed benchmark input asset.\')\\n        if name not in tables:\\n            path = work/name\\n            if path.is_file():\\n                verify_file(path, asset[\'sha256\'])\\n            else:\\n                metadata = source.relative(asset[\'source_path\'])\\n                if metadata is None:\\n                    raise FileNotFoundError(\'Saved benchmark text is missing from the input-library directory.\')\\n                temporary = path.with_suffix(\'.partial\')\\n                source.download(metadata, temporary, expected_sha256=asset[\'sha256\'], max_bytes=64*1024*1024)\\n                temporary.replace(path)\\n            table = pq.read_table(path, columns=[\'id\', \'text\'])\\n            if len(table) != asset[\'rows\']:\\n                raise ValueError(\'Benchmark asset row coverage changed.\')\\n            tables[name] = table\\n        rows = tables[name].slice(task[\'offset\'], task[\'rows\']).to_pylist()\\n        if (len(rows) != task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows)) != task[\'keys_digest\']\\n                or any(r[\'id\'] != key(r[\'text\']) for r in rows)):\\n            raise ValueError(\'Benchmark input text/key binding changed.\')\\n        count = min(96, len(rows))\\n        for i in range(count):\\n            row = rows[i*(len(rows)-1)//max(1,count-1)]\\n            values.setdefault(row[\'id\'], row[\'text\'])\\n        provenance.append({\'task\': task[\'id\'], \'asset\': name, \'source_sha256\': task[\'sha256\']})\\n        print(\'COLAB_BENCHMARK_SAMPLE:\',len(tables),\'/3 assets |\',len(values),\'unique texts\',flush=True)\\n    items = list(values.items())\\n    calibration, evaluation = items[::2], items[1::2]\\n    if min(len(calibration), len(evaluation)) < 32:\\n        raise ValueError(\'Not enough independent calibration/evaluation texts; no model was loaded.\')\\n    result = seal({\'schema\':\'qwen-colab-sample-v1\',\'policy\':SAMPLE_POLICY,\'job\':job[\'manifest_sha256\'],\\n        \'calibration\':[{\'id\':k,\'text\':v} for k,v in calibration],\\n        \'evaluation\':[{\'id\':k,\'text\':v} for k,v in evaluation],\\n        \'source_tasks\':provenance,\'assets_read\':len(tables),\'tasks_committed\':0})\\n    atomic_json(sample_path, result)\\n    return sample_path, result\\n\\n\\ndef compare_execution(native, tuned, sample, cuda, *, clock=time.perf_counter, report=print):\\n    \\"\\"\\"Alternate two held-out native/tuned passes with CUDA synchronization.\\"\\"\\"\\n    import numpy as np\\n    try:\\n        from .qwen_token_batching import vector_agreement\\n    except ImportError:\\n        from qwen_token_batching import vector_agreement\\n    from vietmedbridge.embeddings import validate_vectors\\n    checked(sample)\\n    calibration, evaluation = sample[\'calibration\'], sample[\'evaluation\']\\n    if (min(len(calibration),len(evaluation)) < 32\\n            or {r[\'id\'] for r in calibration} & {r[\'id\'] for r in evaluation}\\n            or any(r[\'id\'] != key(r[\'text\']) for r in [*calibration,*evaluation])):\\n        raise ValueError(\'Calibration and evaluation must have disjoint validated text IDs.\')\\n    calibration = [r[\'text\'] for r in calibration]\\n    evaluation = [r[\'text\'] for r in evaluation]\\n    lengths = tuned.lengths(calibration)\\n    eval_lengths = tuned.lengths(evaluation)\\n    tuned._calibrate(calibration, lengths)\\n    native.encode(evaluation[:4], batch_size=16)\\n    cuda.synchronize()\\n    runs, agreements = [], []\\n    for repeat, order in enumerate(((\'native\',\'tuned\'),(\'tuned\',\'native\'))):\\n        vectors = {}\\n        for mode in order:\\n            cuda.synchronize()\\n            cuda.reset_peak_memory_stats()\\n            start = clock()\\n            value = native.encode(evaluation,batch_size=16) if mode == \'native\' else tuned.encode(evaluation,batch_size=64)\\n            cuda.synchronize()\\n            elapsed = max(clock()-start,1e-9)\\n            validate_vectors(np.asarray(value),len(evaluation),native.dimension)\\n            vectors[mode] = value\\n            item = {\'repeat\':repeat,\'mode\':mode,\'seconds\':elapsed,\'texts_per_second\':len(evaluation)/elapsed,\\n                    \'tokens_per_second\':sum(eval_lengths)/elapsed,\\n                    \'peak_allocated_bytes\':cuda.max_memory_allocated(),\'peak_reserved_bytes\':cuda.max_memory_reserved()}\\n            runs.append(item)\\n            report(\'COLAB_BENCHMARK_PASS: \'+json.dumps(item),flush=True)\\n        compatible, error = vector_agreement(vectors[\'native\'],vectors[\'tuned\'])\\n        agreements.append({\'repeat\':repeat,\'compatible\':compatible,**error})\\n    baseline = statistics.median(r[\'seconds\'] for r in runs if r[\'mode\'] == \'native\')\\n    candidate = statistics.median(r[\'seconds\'] for r in runs if r[\'mode\'] == \'tuned\')\\n    consistent = all(next(r[\'seconds\'] for r in runs if r[\'repeat\']==i and r[\'mode\']==\'tuned\')\\n                     <= next(r[\'seconds\'] for r in runs if r[\'repeat\']==i and r[\'mode\']==\'native\')/1.05 for i in range(2))\\n    accepted = bool(tuned.selected is not None and consistent and all(a[\'compatible\'] for a in agreements))\\n    return {\'calibration_texts\':len(calibration),\'evaluation_texts\':len(evaluation),\\n            \'evaluation_tokens\':sum(eval_lengths),\'evaluation_token_range\':[min(eval_lengths),max(eval_lengths)],\\n            \'native_texts_per_second\':len(evaluation)/baseline,\'candidate_texts_per_second\':len(evaluation)/candidate,\\n            \'held_out_speedup\':baseline/candidate,\'candidate_accepted\':accepted,\'runs\':runs,\'agreements\':agreements,\\n            \'recommendation\':\'TOKEN_BATCHES_VALIDATED_ON_SAMPLE\' if accepted else \'KEEP_NATIVE_FOR_THIS_SAMPLE\'}\\n\\n\\ndef run_benchmark(job_path, sample_path, checkout, output, *, force=False):\\n    import torch\\n    try:\\n        from .qwen_token_batching import TokenBatchEncoder\\n    except ImportError:\\n        from qwen_token_batching import TokenBatchEncoder\\n    from vietmedbridge.qwen_models import TorchQwenEncoder\\n    from vietmedbridge.qwen_execution import configure_embedding_execution\\n    job, sample = validate_job(read_json(job_path)), checked(read_json(sample_path))\\n    if sample[\'job\'] != job[\'manifest_sha256\']:\\n        raise ValueError(\'Sample and job differ.\')\\n    if not torch.cuda.is_available() or benchmark_runtime() != job[\'runtime\']:\\n        raise RuntimeError(\'Select Colab GPU with the exact original CUDA/runtime before benchmarking.\')\\n    for name, checksum in job[\'model_code\'].items():\\n        verify_file(Path(checkout)/\'src/vietmedbridge\'/name,checksum)\\n    gpu = torch.cuda.get_device_name(0)\\n    binding = {\'job\':job[\'manifest_sha256\'],\'sample\':sample[\'manifest_sha256\'],\'runtime\':job[\'runtime\'],\\n               \'gpu\':gpu,\'benchmark_code_sha256\':sha256_file(__file__),\\n               \'tuning_code_sha256\':sha256_file(Path(__file__).with_name(\'qwen_token_batching.py\'))}\\n    output = Path(output)\\n    if output.is_file() and not force:\\n        previous = checked(read_json(output))\\n        if previous.get(\'binding\') == binding and previous.get(\'state\') == \'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\':\\n            print(\'BENCHMARK_REUSED_SAVED_MEASUREMENT:\',str(output),\'| no GPU work repeated\',flush=True)\\n            return previous\\n    owner, stage = None, \'model_load\'\\n    try:\\n        loading_started = time.perf_counter()\\n        owner = TorchQwenEncoder(job[\'config\'][\'second_dense\'])\\n        model_load_seconds = time.perf_counter()-loading_started\\n        native = owner.for_role(\'corpus\')\\n        expected = job[\'models\'][\'qwen\']\\n        if native.identity != expected[\'encoder\'] or native.dimension != expected[\'dimension\']:\\n            raise ValueError(\'Benchmark encoder differs from existing corpus vectors.\')\\n        configure_embedding_execution(native)\\n        stage = \'calibration_and_evaluation\'\\n        tuned = TokenBatchEncoder(native,output.with_name(\'tuning.json\'),attention_tuning=True,\\n                                 deadline=time.monotonic()+1200)\\n        evaluation = compare_execution(native,tuned,sample,torch.cuda)\\n        is_t4 = bool(re.search(r\'\\\\bT4\\\\b\',gpu,re.I))\\n        result = seal({\'state\':\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',\'binding\':binding,\\n            \'gpu\':gpu,\'compute_capability\':list(torch.cuda.get_device_capability(0)),\\n            \'total_vram_bytes\':torch.cuda.get_device_properties(0).total_memory,\'model_load_seconds\':model_load_seconds,\\n            \'vram_metric\':\'Peak allocated includes the resident model; reserved also includes allocator cache.\',\\n            \'kaggle_t4_hardware_match\':is_t4,\'full_corpus_performance_certified\':False,\\n            \'evaluation\':evaluation,\'calibration\':read_json(tuned.path),\\n            \'source_tasks\':sample[\'source_tasks\'],\'tasks_committed\':0,\'vectors_published\':0,\\n            \'updated_at\':utc_now()})\\n        atomic_json(output,result)\\n        print(\'COLAB_QWEN_BENCHMARK_RESULT:\',json.dumps({\\n            \'gpu\':gpu,\'kaggle_t4_hardware_match\':is_t4,**{k:v for k,v in evaluation.items() if k not in (\'runs\',\'agreements\')},\\n            \'batch_cap\':tuned.safe_batch_size,\'kernel\':tuned.selected_kernel,\'tasks_committed\':0}),flush=True)\\n        return result\\n    except Exception as error:\\n        atomic_json(output,seal({\'state\':\'BENCHMARK_FAILED\',\'binding\':binding,\'gpu\':gpu,\\n            \'stage\':stage,\'error_type\':type(error).__name__,\'tasks_committed\':0,\'vectors_published\':0,\'updated_at\':utc_now()}))\\n        raise\\n    finally:\\n        if owner is not None:\\n            owner.close()\\n\\n\\ndef publish_report(source, path, *, files=None):\\n    \\"\\"\\"Create one report under the project staging folder; never update other data.\\"\\"\\"\\n    path = Path(path)\\n    value = checked(read_json(path))\\n    if (value.get(\'state\') not in (\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',\'BENCHMARK_FAILED\')\\n            or value.get(\'tasks_committed\') != 0 or value.get(\'vectors_published\') != 0\\n            or value.get(\'binding\',{}).get(\'job\') != LIBRARY or path.stat().st_size > 1024*1024):\\n        raise ValueError(\'Only a bounded benchmark report can be published.\')\\n    json.dumps(value,allow_nan=False)\\n    if files is None:\\n        import copy\\n        from googleapiclient.discovery import build\\n        files = build(\'drive\',\'v3\',credentials=copy.copy(source._session().credentials),cache_discovery=False).files()\\n    from googleapiclient.http import MediaIoBaseUpload\\n    fields = \'id,name,mimeType,size,md5Checksum,parents,webViewLink\'\\n    folder = source.child(STAGING,\'qwen_performance_benchmarks\',folder=True)\\n    if folder is None:\\n        identifier = files.generateIds(count=1,space=\'drive\',type=\'files\').execute(num_retries=3)[\'ids\'][0]\\n        try:\\n            folder = files.create(body={\'id\':identifier,\'name\':\'qwen_performance_benchmarks\',\'parents\':[STAGING],\\n                \'mimeType\':\'application/vnd.google-apps.folder\'},fields=fields,supportsAllDrives=True).execute(num_retries=3)\\n        except Exception:\\n            folder = files.get(fileId=identifier,fields=fields,supportsAllDrives=True).execute(num_retries=3)\\n        if folder.get(\'name\') != \'qwen_performance_benchmarks\' or folder.get(\'parents\') != [STAGING]:\\n            raise ValueError(\'Benchmark destination differs from the project staging folder.\')\\n    name = \'qwen-benchmark-\'+value[\'manifest_sha256\']+\'.json\'\\n    existing = source.child(folder[\'id\'],name)\\n    payload = path.read_bytes()\\n    md5 = hashlib.md5(payload).hexdigest()\\n    if existing is None:\\n        identifier = files.generateIds(count=1,space=\'drive\',type=\'files\').execute(num_retries=3)[\'ids\'][0]\\n        media = MediaIoBaseUpload(io.BytesIO(payload),mimetype=\'application/json\',resumable=False)\\n        try:\\n            existing = files.create(body={\'id\':identifier,\'name\':name,\'parents\':[folder[\'id\']]},media_body=media,\\n                fields=fields,supportsAllDrives=True).execute(num_retries=3)\\n        except Exception:\\n            existing = files.get(fileId=identifier,fields=fields,supportsAllDrives=True).execute(num_retries=3)\\n    saved = files.get(fileId=existing[\'id\'],fields=fields,supportsAllDrives=True).execute(num_retries=3)\\n    if (saved.get(\'name\') != name or saved.get(\'parents\') != [folder[\'id\']]\\n            or saved.get(\'md5Checksum\') != md5 or int(saved.get(\'size\',-1)) != len(payload)):\\n        raise ValueError(\'Drive benchmark report readback differs from the measured result.\')\\n    print(\'BENCHMARK_SAVED_ON_DRIVE:\',saved.get(\'webViewLink\',saved[\'id\']),flush=True)\\n    return saved\\n\\n\\nif __name__ == \'__main__\':\\n    parser = argparse.ArgumentParser()\\n    for name in (\'job\',\'sample\',\'checkout\',\'output\'):\\n        parser.add_argument(\'--\'+name,required=True)\\n    parser.add_argument(\'--force\',action=\'store_true\')\\n    args = parser.parse_args()\\n    run_benchmark(args.job,args.sample,args.checkout,args.output,force=args.force)\\n", "qwen_t4_experiments.py": "\\"\\"\\"Benchmark-only execution trials; the live Kaggle v2 worker stays unchanged.\\"\\"\\"\\nfrom contextlib import contextmanager\\nfrom pathlib import Path\\nimport argparse\\nimport json\\nimport re\\nimport statistics\\nimport time\\n\\nimport numpy as np\\n\\nfrom vietmedbridge.artifacts import atomic_json,read_json,sha256_file,utc_now,verify_file\\nfrom vietmedbridge.content_embeddings import checked,seal\\nfrom vietmedbridge.embeddings import validate_vectors\\ntry:\\n    from .qwen_token_batching import TokenBatchEncoder,vector_agreement\\n    from .colab_qwen_benchmark import validate_job,benchmark_runtime\\nexcept ImportError:\\n    from qwen_token_batching import TokenBatchEncoder,vector_agreement\\n    from colab_qwen_benchmark import validate_job,benchmark_runtime\\n\\nPOLICY=\'qwen-t4-extra-trials-v1\'\\nCONFIGS=(\\n    {\'name\':\'auto-b8\',\'cap\':8,\'backend\':\'auto\',\'threads\':None},\\n    {\'name\':\'auto-b12\',\'cap\':12,\'backend\':\'auto\',\'threads\':None},\\n    {\'name\':\'auto-b24\',\'cap\':24,\'backend\':\'auto\',\'threads\':None},\\n    {\'name\':\'efficient-b16\',\'cap\':16,\'backend\':\'efficient\',\'threads\':None},\\n    {\'name\':\'efficient-b24\',\'cap\':24,\'backend\':\'efficient\',\'threads\':None},\\n    {\'name\':\'threads1-b16\',\'cap\':16,\'backend\':\'auto\',\'threads\':1},\\n    {\'name\':\'threads1-efficient-b16\',\'cap\':16,\'backend\':\'efficient\',\'threads\':1},\\n)\\n\\n\\n@contextmanager\\ndef efficient_attention():\\n    from transformers.integrations import sdpa_attention as integration\\n    from torch.nn.attention import SDPBackend,sdpa_kernel\\n    original=integration.use_gqa_in_sdpa\\n    try:\\n        # Equivalent K/V head expansion lets the efficient backend be tried\\n        # on torch 2.8, where fused GQA is otherwise restricted to flash/math.\\n        # Masks, causality and all model/tokenizer parameters are untouched.\\n        integration.use_gqa_in_sdpa=lambda *a,**k:False\\n        with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):\\n            yield\\n    finally:\\n        integration.use_gqa_in_sdpa=original\\n\\n\\n@contextmanager\\ndef execution_context(torch,config):\\n    if config not in CONFIGS:\\n        raise ValueError(\'Unknown benchmark execution configuration.\')\\n    threads=config[\'threads\']\\n    previous=torch.get_num_threads() if threads is not None else None\\n    try:\\n        if threads is not None:torch.set_num_threads(threads)\\n        if config[\'backend\']==\'efficient\':\\n            with efficient_attention():yield\\n        else:yield\\n    finally:\\n        if previous is not None:torch.set_num_threads(previous)\\n\\n\\ndef _known_unavailable(error,torch):\\n    if isinstance(error,torch.OutOfMemoryError):return True\\n    return isinstance(error,RuntimeError) and any(s in str(error).lower() for s in\\n        (\'no available kernel\',\'no kernel found\',\'no viable backend\',\'not compiled with\',\'requires sm\',\'does not support this\'))\\n\\n\\ndef run_suite(native,reference,sample,cuda,*,clock=time.perf_counter,report=print):\\n    checked(sample)\\n    calibration,evaluation=sample[\'calibration\'],sample[\'evaluation\']\\n    from vietmedbridge.content_embeddings import key\\n    if ({r[\'id\'] for r in calibration}&{r[\'id\'] for r in evaluation}\\n            or min(len(calibration),len(evaluation))<32\\n            or any(r[\'id\']!=key(r[\'text\']) for r in [*calibration,*evaluation])):\\n        raise ValueError(\'Independent validated calibration/evaluation texts are required.\')\\n    calibration=[r[\'text\'] for r in calibration]\\n    evaluation=[r[\'text\'] for r in evaluation]\\n    reference._calibrate(calibration,reference.lengths(calibration))\\n    count=min(64,len(calibration))\\n    trial_texts=[calibration[i*(len(calibration)-1)//max(1,count-1)] for i in range(count)]\\n    runners={}\\n    owner=getattr(native,\'encoder\',native)\\n    def variant(config,texts):\\n        name=config[\'name\']\\n        if name not in runners:\\n            runners[name]=TokenBatchEncoder(native,reference.path.with_name(name+\'.json\'),report=report)\\n        runner=runners[name]\\n        lengths=runner.lengths(texts)\\n        with execution_context(owner.torch,config):\\n            return runner._run(texts,lengths,config[\'cap\'],\'native\',24_576)\\n    def measure(operation,rows):\\n        cuda.synchronize();cuda.reset_peak_memory_stats()\\n        started=clock()\\n        value=np.asarray(operation(),dtype=np.float32)\\n        cuda.synchronize()\\n        elapsed=max(clock()-started,1e-9)\\n        validate_vectors(value,rows,native.dimension)\\n        return value,elapsed,{\'peak_allocated_bytes\':cuda.max_memory_allocated(),\\n                              \'peak_reserved_bytes\':cuda.max_memory_reserved()}\\n    frozen,_,_=measure(lambda:native.encode(trial_texts,batch_size=16),len(trial_texts))\\n    _,reference_seconds,_=measure(lambda:reference.encode(trial_texts,batch_size=64),len(trial_texts))\\n    best,best_seconds,trials=None,reference_seconds,[]\\n    started=clock()\\n    for config in CONFIGS:\\n        if clock()-started>240:\\n            trials.append({\'name\':config[\'name\'],\'state\':\'SKIPPED_TIME_BUDGET\'})\\n            continue\\n        if config[\'threads\']==1 and owner.torch.get_num_threads()==1:\\n            trials.append({\'name\':config[\'name\'],\'state\':\'SKIPPED_ALREADY_ONE_THREAD\'})\\n            continue\\n        try:\\n            variant(config,trial_texts[:4])  # backend warm-up is outside timing\\n            value,seconds,memory=measure(lambda:variant(config,trial_texts),len(trial_texts))\\n            compatible,agreement=vector_agreement(frozen,value)\\n            item={**config,\'state\':\'MEASURED\',\'seconds\':seconds,\'compatible\':compatible,\\n                  \'speedup_over_v2\':reference_seconds/seconds,**agreement,**memory,\\n                  \'oom_backoffs\':runners[config[\'name\']].ooms}\\n            if compatible and seconds<best_seconds and seconds<=reference_seconds/1.05:\\n                best,best_seconds=config,seconds\\n        except (RuntimeError,ValueError) as error:\\n            if isinstance(error,RuntimeError) and not _known_unavailable(error,owner.torch):raise\\n            item={**config,\'state\':\'UNAVAILABLE\' if isinstance(error,RuntimeError) else \'INVALID_VECTORS\',\\n                  \'error_type\':type(error).__name__}\\n        trials.append(item)\\n        report(\'T4_EXTRA_TRIAL: \'+json.dumps(item),flush=True)\\n    if best is None:\\n        return {\'measurement_scope\':\'calibration_only_no_faster_candidate\',\'candidate_accepted\':False,\\n                \'reference_v2_texts_per_second\':len(trial_texts)/reference_seconds,\\n                \'candidate_texts_per_second\':len(trial_texts)/reference_seconds,\\n                \'additional_speedup_over_v2\':1.0,\'selected_config\':None,\'trials\':trials,\\n                \'evaluation_texts\':0,\'recommendation\':\'KEEP_EXISTING_V2\'}\\n    runs,agreements=[],[]\\n    for repeat,order in enumerate(((\'original_native\',\'reference_v2\',\'candidate\'),\\n                                   (\'candidate\',\'reference_v2\',\'original_native\'))):\\n        vectors={}\\n        for mode in order:\\n            operation=(lambda:native.encode(evaluation,batch_size=16)) if mode==\'original_native\' else (\\n                (lambda:reference.encode(evaluation,batch_size=64)) if mode==\'reference_v2\' else\\n                (lambda:variant(best,evaluation)))\\n            try:\\n                value,seconds,memory=measure(operation,len(evaluation))\\n            except (RuntimeError,ValueError) as error:\\n                if mode!=\'candidate\' or (isinstance(error,RuntimeError) and not _known_unavailable(error,owner.torch)):\\n                    raise\\n                # A backend that fails on held-out shapes cannot be recommended.\\n                # Do not turn a partial timing into a claimed speedup.\\n                return {\'measurement_scope\':\'held_out_candidate_failed\',\'candidate_accepted\':False,\\n                    \'reference_v2_texts_per_second\':len(trial_texts)/reference_seconds,\\n                    \'candidate_texts_per_second\':None,\'additional_speedup_over_v2\':None,\\n                    \'selected_config\':None,\'best_calibration_candidate\':best,\\n                    \'evaluation_texts\':len(evaluation),\'trials\':trials,\'runs\':runs,\\n                    \'error_type\':type(error).__name__,\'recommendation\':\'KEEP_EXISTING_V2\'}\\n            vectors[mode]=value\\n            item={\'repeat\':repeat,\'mode\':mode,\'seconds\':seconds,\'texts_per_second\':len(evaluation)/seconds,**memory}\\n            runs.append(item)\\n            report(\'T4_EXTRA_PASS: \'+json.dumps(item),flush=True)\\n        compatible,agreement=vector_agreement(vectors[\'original_native\'],vectors[\'candidate\'])\\n        agreements.append({\'repeat\':repeat,\'compatible\':compatible,**agreement})\\n    def seconds(mode):return statistics.median(r[\'seconds\'] for r in runs if r[\'mode\']==mode)\\n    consistent=all(next(r[\'seconds\'] for r in runs if r[\'repeat\']==i and r[\'mode\']==\'candidate\')\\n        <=next(r[\'seconds\'] for r in runs if r[\'repeat\']==i and r[\'mode\']==\'reference_v2\')/1.05 for i in range(2))\\n    accepted=consistent and all(a[\'compatible\'] for a in agreements)\\n    return {\'measurement_scope\':\'held_out_two_passes_against_existing_v2\',\'candidate_accepted\':accepted,\\n            \'reference_v2_texts_per_second\':len(evaluation)/seconds(\'reference_v2\'),\\n            \'original_native_texts_per_second\':len(evaluation)/seconds(\'original_native\'),\\n            \'candidate_texts_per_second\':len(evaluation)/seconds(\'candidate\'),\\n            \'additional_speedup_over_v2\':seconds(\'reference_v2\')/seconds(\'candidate\'),\\n            \'selected_config\':best if accepted else None,\'best_calibration_candidate\':best,\\n            \'evaluation_texts\':len(evaluation),\'trials\':trials,\'runs\':runs,\'agreements\':agreements,\\n            \'recommendation\':\'NEW_CANDIDATE_VALIDATED_ON_SAMPLE\' if accepted else \'KEEP_EXISTING_V2\'}\\n\\n\\ndef run(job_path,sample_path,checkout,output,*,force=False):\\n    import torch\\n    from vietmedbridge.qwen_models import TorchQwenEncoder\\n    from vietmedbridge.qwen_execution import configure_embedding_execution\\n    job,sample=validate_job(read_json(job_path)),checked(read_json(sample_path))\\n    if sample[\'job\']!=job[\'manifest_sha256\'] or not torch.cuda.is_available() or benchmark_runtime()!=job[\'runtime\']:\\n        raise ValueError(\'Original sample and CUDA runtime are required.\')\\n    for name,checksum in job[\'model_code\'].items():verify_file(Path(checkout)/\'src/vietmedbridge\'/name,checksum)\\n    gpu=torch.cuda.get_device_name(0)\\n    binding={\'job\':job[\'manifest_sha256\'],\'sample\':sample[\'manifest_sha256\'],\'runtime\':job[\'runtime\'],\'gpu\':gpu,\\n             \'policy\':POLICY,\'experiment_code_sha256\':sha256_file(__file__),\\n             \'tuning_code_sha256\':sha256_file(Path(__file__).with_name(\'qwen_token_batching.py\'))}\\n    output=Path(output)\\n    if output.is_file() and not force:\\n        previous=checked(read_json(output))\\n        if previous.get(\'binding\')==binding and previous.get(\'state\')==\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\':\\n            print(\'T4_EXTRA_REUSED_SAVED_MEASUREMENT:\',str(output),flush=True)\\n            return previous\\n    owner=None\\n    try:\\n        owner=TorchQwenEncoder(job[\'config\'][\'second_dense\'])\\n        native=owner.for_role(\'corpus\')\\n        if native.identity!=job[\'models\'][\'qwen\'][\'encoder\'] or native.dimension!=job[\'models\'][\'qwen\'][\'dimension\']:\\n            raise ValueError(\'Native model differs from the existing vector producer.\')\\n        configure_embedding_execution(native)\\n        reference=TokenBatchEncoder(native,output.with_name(\'reference-v2-tuning.json\'),attention_tuning=True,\\n                                   deadline=time.monotonic()+1200)\\n        evaluation=run_suite(native,reference,sample,torch.cuda)\\n        result=seal({\'state\':\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',\'binding\':binding,\'gpu\':gpu,\\n            \'kaggle_t4_hardware_match\':bool(re.search(r\'\\\\bT4\\\\b\',gpu,re.I)),\'full_corpus_performance_certified\':False,\\n            \'evaluation\':evaluation,\'calibration\':read_json(reference.path),\'source_tasks\':sample[\'source_tasks\'],\\n            \'tasks_committed\':0,\'vectors_published\':0,\'production_worker_changed\':False,\'updated_at\':utc_now()})\\n        atomic_json(output,result)\\n        print(\'T4_EXTRA_RESULT:\',json.dumps(evaluation),flush=True)\\n        return result\\n    except Exception as error:\\n        atomic_json(output,seal({\'state\':\'BENCHMARK_FAILED\',\'binding\':binding,\'gpu\':gpu,\\n            \'error_type\':type(error).__name__,\'tasks_committed\':0,\'vectors_published\':0,\'updated_at\':utc_now()}))\\n        raise\\n    finally:\\n        if owner is not None:owner.close()\\n\\n\\nif __name__==\'__main__\':\\n    parser=argparse.ArgumentParser()\\n    for name in (\'job\',\'sample\',\'checkout\',\'output\'):parser.add_argument(\'--\'+name,required=True)\\n    parser.add_argument(\'--force\',action=\'store_true\')\\n    args=parser.parse_args()\\n    run(args.job,args.sample,args.checkout,args.output,force=args.force)\\n"}')

for name,source in SOURCES.items():
    (WORK/name).write_text(source,encoding='utf-8')
    sys.modules.pop(Path(name).stem,None)
importlib.invalidate_caches()
from new_data_drive_api import DriveReceiptSource
api=DriveReceiptSource(lambda:AuthorizedSession(copy.copy(credentials)),WORK,WORK,
    data_folder_id='1NJ9LtdOSRLZtCvBuML7O8UYppiJuIy4g')
JOB_PATH=WORK/'job.json'
def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def read_library():
    value=json.loads(JOB_PATH.read_text(encoding='utf-8'))
    if (value.get('manifest_sha256')!='d6ab9a432ac64e5aadb39a826b7dc2176094cc66d4f4e3d9e17c121e071c0711'
        or digest({k:v for k,v in value.items() if k!='manifest_sha256'})!=value['manifest_sha256']):
        raise ValueError('Thư viện benchmark không đúng job đã freeze.')
    return value
if not JOB_PATH.is_file():
    print('BENCHMARK_SETUP: read one saved library JSON; no corpus/vector inventory',flush=True)
    metadata=api.relative('input_library.json')
    if metadata is None: raise FileNotFoundError('Tài khoản Google chưa đọc được thư viện trong VietMedBridge.')
    api.download(metadata,WORK/'job.partial',max_bytes=32*1024*1024)
    (WORK/'job.partial').replace(JOB_PATH)
JOB=read_library()
if (JOB.get('repo_url')!='https://github.com/Platypus27-coder/VietMedBridge.git'
    or not re.fullmatch('[0-9a-f]{40}',JOB.get('code_commit',''))):
    raise ValueError('Producer repo/commit không hợp lệ.')
CHECKOUT=Path('/content/VietMedBridge-qwen-benchmark')
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',JOB['repo_url'],str(CHECKOUT)],check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'],text=True).strip()!=JOB['repo_url']:
    raise ValueError('Checkout thuộc repo khác.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',JOB['code_commit']],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'],check=True)
runtime=JOB['runtime']
if (set(runtime)!={'torch','transformers','bitsandbytes'}
    or any(not re.fullmatch('[0-9][A-Za-z0-9.+_-]*',v) for v in runtime.values())):
    raise ValueError('Runtime lock không hợp lệ.')
try: installed=importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError: installed=None
if installed!=runtime['torch']:
    match=re.search(r'\+(cu[0-9]+)$',runtime['torch'])
    if match is None: raise ValueError('Cần bản torch CUDA gốc của Kaggle.')
    print('BENCHMARK_SETUP: install original CUDA torch',runtime['torch'],flush=True)
    subprocess.run([sys.executable,'-m','pip','install','torch=='+runtime['torch'],
        '--index-url','https://download.pytorch.org/whl/'+match[1]],check=True)
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
print('BENCHMARK_SETUP: install model-only dependencies; no crawl/index packages',flush=True)
subprocess.run([sys.executable,'-m','pip','install','numpy','tqdm','pyarrow==21.0.0',
    'transformers=='+runtime['transformers'],'bitsandbytes=='+runtime['bitsandbytes'],
    'tokenizers==0.22.2','huggingface-hub==0.36.0','accelerate==1.12.0',
    'google-api-python-client'],check=True)
env=os.environ.copy()
env['PYTHONPATH']=os.pathsep.join((str(WORK),str(CHECKOUT/'src')))
env['HF_HOME']=str(WORK/'hf_cache')
env['TOKENIZERS_PARALLELISM']='false'
env['CUDA_VISIBLE_DEVICES']='0'
probe="import torch; from transformers import AutoModel; print('BENCHMARK_CUDA:',torch.__version__,torch.cuda.is_available(),flush=True); assert torch.cuda.is_available()"
checked_probe=subprocess.run([sys.executable,'-u','-c',probe],env=env,text=True,capture_output=True)
if checked_probe.returncode and any(x in checked_probe.stderr.lower() for x in ('torchvision','torchaudio','torchvision::nms')):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
    checked_probe=subprocess.run([sys.executable,'-u','-c',probe],env=env,text=True,capture_output=True)
print(checked_probe.stdout,flush=True)
if checked_probe.returncode:
    print(checked_probe.stderr,flush=True)
    raise RuntimeError('CUDA/runtime chưa sẵn sàng; chưa nạp model hoặc chạy benchmark.')
sys.path.insert(0,str(CHECKOUT/'src'))


In [ ]:
#@title 2. Lấy mẫu nhỏ từ tối đa ba file text trên Drive
importlib.invalidate_caches()
from colab_qwen_benchmark import prepare_sample,validate_job
validate_job(JOB)
SAMPLE_PATH,SAMPLE=prepare_sample(api,JOB,WORK/'sample')
print('BENCHMARK_SAMPLE_READY: calibration',len(SAMPLE['calibration']),
      '| held-out evaluation',len(SAMPLE['evaluation']),'| assets',SAMPLE['assets_read'],flush=True)

In [ ]:
#@title 3. Đo một GPU: native batch 16 so với bộ tự chọn Qwen v2
REPORT_PATH=WORK/'benchmark.json'
command=[sys.executable,'-u',str(WORK/'colab_qwen_benchmark.py'),
    '--job',str(JOB_PATH),'--sample',str(SAMPLE_PATH),'--checkout',str(CHECKOUT),'--output',str(REPORT_PATH)]
if FORCE_REMEASURE: command.append('--force')
started=time.monotonic()
process=subprocess.Popen(command,env=env)
try:
    while process.poll() is None:
        if time.monotonic()-started>2400:
            raise RuntimeError('Benchmark vượt 40 phút kể cả tải model; giữ runtime để kiểm tra cache.')
        print('COLAB_BENCHMARK_HEARTBEAT:',int(time.monotonic()-started),'s',flush=True)
        time.sleep(15)
finally:
    if process.poll() is None:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill();process.wait(timeout=10)
if not REPORT_PATH.is_file():
    raise RuntimeError('Chưa có báo cáo benchmark. Xem traceback; không có task/corpus nào bị thay đổi.')
result=json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print('BENCHMARK_STATE:',result['state'],'| tasks committed',result['tasks_committed'],flush=True)
if process.returncode and result['state']!='BENCHMARK_FAILED':
    raise RuntimeError('Tiến trình lỗi; không dùng báo cáo cũ để kết luận lượt này.')

In [ ]:
#@title 4. Lưu báo cáo nhỏ lên Drive — có thể chạy lại riêng cell này
from colab_qwen_benchmark import publish_report
DRIVE_REPORT=publish_report(api,REPORT_PATH)
result=json.loads(REPORT_PATH.read_text(encoding='utf-8'))
if result['state']=='BENCHMARK_COMPLETE_NO_TASKS_COMMITTED':
    e=result['evaluation']
    print('BENCHMARK_SUMMARY: native',round(e['native_texts_per_second'],2),'texts/s | candidate',
        round(e['candidate_texts_per_second'],2),'texts/s | held-out speedup',round(e['held_out_speedup'],2),
        '| accepted',e['candidate_accepted'],'| T4 match',result['kaggle_t4_hardware_match'],flush=True)
else:
    print('BENCHMARK_FAILED: báo cáo lỗi đã lưu Drive; chưa kết luận tốc độ.',flush=True)

In [ ]:
#@title 5. Thử thêm batch 8/12/24, CPU threads và efficient attention — so với v2
EXTRA_REPORT_PATH=WORK/'benchmark-extended.json'
command=[sys.executable,'-u',str(WORK/'qwen_t4_experiments.py'),
    '--job',str(JOB_PATH),'--sample',str(SAMPLE_PATH),'--checkout',str(CHECKOUT),'--output',str(EXTRA_REPORT_PATH)]
if FORCE_REMEASURE: command.append('--force')
started=time.monotonic()
process=subprocess.Popen(command,env=env)
try:
    while process.poll() is None:
        if time.monotonic()-started>2400:
            raise RuntimeError('Benchmark vượt 40 phút kể cả tải model; giữ runtime để kiểm tra cache.')
        print('COLAB_BENCHMARK_HEARTBEAT:',int(time.monotonic()-started),'s',flush=True)
        time.sleep(15)
finally:
    if process.poll() is None:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill();process.wait(timeout=10)
if not EXTRA_REPORT_PATH.is_file():
    raise RuntimeError('Chưa có báo cáo benchmark. Xem traceback; không có task/corpus nào bị thay đổi.')
result=json.loads(EXTRA_REPORT_PATH.read_text(encoding='utf-8'))
print('BENCHMARK_STATE:',result['state'],'| tasks committed',result['tasks_committed'],flush=True)
if process.returncode and result['state']!='BENCHMARK_FAILED':
    raise RuntimeError('Tiến trình lỗi; không dùng báo cáo cũ để kết luận lượt này.')

In [ ]:
#@title 6. Lưu kết quả bổ sung; chỉ nhận mức tăng thêm so với v2
EXTRA_DRIVE_REPORT=publish_report(api,EXTRA_REPORT_PATH)
extra=json.loads(EXTRA_REPORT_PATH.read_text(encoding='utf-8'))
if extra['state']=='BENCHMARK_COMPLETE_NO_TASKS_COMMITTED':
    e=extra['evaluation']
    print('T4_EXTRA_SUMMARY: existing v2',round(e['reference_v2_texts_per_second'],2),'texts/s | candidate',
        (round(e['candidate_texts_per_second'],2) if e['candidate_texts_per_second'] is not None else 'unavailable'),
        'texts/s | additional speedup',
        (round(e['additional_speedup_over_v2'],2) if e['additional_speedup_over_v2'] is not None else 'unavailable'),
        '| accepted',e['candidate_accepted'],'| scope',e['measurement_scope'],
        '| config',e['selected_config'],flush=True)
else:
    print('T4_EXTRA_FAILED: giữ v2; báo cáo lỗi đã lưu Drive.',flush=True)